# CCT 依赖打包

> **Internet ON**, 无需 GPU (用 CPU 即可)

下载所有 CCT 训练所需的 pip 包并打包为 tar.gz，之后上传为 Kaggle Dataset 供离线训练使用。

In [ ]:
import subprocess, sys, os, shutil

WORK = '/tmp/cct_wheels'
OUT = '/kaggle/working'

# 清理旧数据
if os.path.exists(WORK):
    shutil.rmtree(WORK)
os.makedirs(WORK, exist_ok=True)

PACKAGES = ['transformers>=4.48.0', 'accelerate>=1.0.0', 'datasets>=3.0.0', 'sentencepiece', 'safetensors']

print('下载依赖 wheels...')
for pkg in PACKAGES:
    print('  下载: %s' % pkg)
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'download',
         '--dest', WORK,
         '--only-binary=:all:',
         pkg],
        capture_output=True, text=True)
    if r.returncode != 0:
        print('    ⚠ binary 失败, 尝试允许 source...')
        subprocess.run(
            [sys.executable, '-m', 'pip', 'download',
             '--dest', WORK, pkg],
            capture_output=True, text=True)

wheels = os.listdir(WORK)
print('\n下载完成: %d 个文件' % len(wheels))
for w in sorted(wheels):
    sz = os.path.getsize(os.path.join(WORK, w)) / 1024 / 1024
    print('  %s (%.1f MB)' % (w, sz))

In [ ]:
import tarfile

TAR_NAME = 'cct_wheels.tar.gz'
tar_path = os.path.join(OUT, TAR_NAME)

print('打包 %s ...' % tar_path)
with tarfile.open(tar_path, 'w:gz') as tar:
    for fname in sorted(os.listdir(WORK)):
        fpath = os.path.join(WORK, fname)
        tar.add(fpath, arcname=fname)

sz_mb = os.path.getsize(tar_path) / 1024 / 1024
print('完成: %s (%.1f MB)' % (tar_path, sz_mb))
print()

# 同时保留散装 wheels 目录供直接使用
wheels_dir = os.path.join(OUT, 'wheels')
if os.path.exists(wheels_dir):
    shutil.rmtree(wheels_dir)
shutil.copytree(WORK, wheels_dir)
print('散装 wheels: %s/' % wheels_dir)

In [ ]:
# 验证: 用离线 wheels 做一次 dry-run 安装
import subprocess, sys

r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--dry-run',
     '--no-index', '--find-links', wheels_dir] + PACKAGES,
    capture_output=True, text=True)

if r.returncode == 0:
    print('✅ 离线安装验证通过!')
    print(r.stdout[-500:] if len(r.stdout) > 500 else r.stdout)
else:
    print('⚠ 验证失败:')
    print(r.stderr[-500:] if len(r.stderr) > 500 else r.stderr)

print()
print('=== 下一步 ===')
print('验证通过后运行下一个 cell 自动上传到 Kaggle')

In [ ]:
# === 上传 wheels 到 Kaggle Dataset ===
import json, os, subprocess, sys

KAGGLE_USER = 'wukeneth'
DS_SLUG = 'cct-wheels'
UPLOAD_DIR = wheels_dir  # from previous cell

# 写 dataset-metadata.json
meta = {
    'title': 'CCT Wheels',
    'id': '%s/%s' % (KAGGLE_USER, DS_SLUG),
    'licenses': [{'name': 'other'}],
}
meta_path = os.path.join(UPLOAD_DIR, 'dataset-metadata.json')
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print('metadata 已写入: %s' % meta_path)

# 检查 dataset 是否已存在
r = subprocess.run(
    ['kaggle', 'datasets', 'list', '-m', '--search', DS_SLUG],
    capture_output=True, text=True)
exists = DS_SLUG in r.stdout

if exists:
    print('Dataset 已存在, 创建新版本...')
    cmd = ['kaggle', 'datasets', 'version', '-p', UPLOAD_DIR,
           '-m', 'auto update wheels', '--dir-mode', 'zip']
else:
    print('创建新 Dataset...')
    cmd = ['kaggle', 'datasets', 'create', '-p', UPLOAD_DIR, '--dir-mode', 'zip']

r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print('ERROR:', r.stderr)
else:
    print('✅ 上传完成! Dataset: https://www.kaggle.com/datasets/%s/%s' % (KAGGLE_USER, DS_SLUG))